# Model Debugging, Inspection, and Modularization

So far, you've focused on building and training models. 
But in the real world, your first attempt at a model rarely works perfectly. 
You'll often encounter cryptic error messages about mismatched tensor shapes, or worse, your model will run without errors but fail to produce meaningful results. 

This is where **debugging, inspection, and modularization** become essential skills. 
In this lab, you'll step into the role of a model investigator. 
You'll start with a broken Convolutional Neural Network (CNN) and use systematic debugging techniques to find and fix the bug. Then, you'll learn how to refactor your code for clarity and reuse, and finally, you'll dissect a complex, pre-trained model to understand its inner workings.

In this lab, you will:

* **Debug** a broken CNN by inserting print statements into the `forward` pass to identify and correct a critical tensor shape mismatch.
* **Refactor** the corrected model using `nn.Sequential` to create a cleaner, more modular, and less error-prone architecture.
* **Inspect** the activation statistics of your model to perform a sanity check for issues like exploding or vanishing gradients.
* **Explore** the architecture of a complex, pre-existing model (`SqueezeNet`) to count its layers and analyze its parameter distribution.

In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.models import SqueezeNet

In [2]:
import helper_utils

## Data Loading

To debug and inspect a model effectively, you'll first need a dataset to work with.
The goal of this lab is to practice an end-to-end debugging and inspection workflow, so you'll use a simple dataset that lets you focus on the model architecture rather than complex data preprocessing.
For this purpose, you'll use the Fashion MNIST dataset, which consists of grayscale images of clothing items and serves as a straightforward benchmark for image classification tasks.
You’ll begin by loading the dataset using PyTorch’s torchvision library, and then create a DataLoader to efficiently handle the data in batches during training and evaluation.

In [3]:
dataset = helper_utils.get_dataset()

transform = transforms.ToTensor()
dataset.transform = transform

Dataset already exists.


In [4]:
batch_size = 64
dataloader = DataLoader(
    dataset=dataset,
    batch_size=batch_size,
    shuffle=False
)

In [6]:
img_batch, label_batch = next(iter(dataloader))
print("Batch shape:", img_batch.shape)

Batch shape: torch.Size([64, 1, 28, 28])


## Debugging through forward pass

When starting to work with a new model, it is common to encounter errors. 
These errors can be due to various reasons, such as incorrect tensor shapes, incompatible operations, or unexpected values.
Sometimes, the model may run without errors but produce incorrect outputs.

In this section, you will explore how to debug a PyTorch model by examining its forward pass.

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block
        self.conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Block
        # For Fashion MNIST: input images are 28x28,
        # after conv+pool: 32x14x14
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 10 classes for Fashion MNIST

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        return x

In [9]:
simple_cnn = SimpleCNN()

try:
    output = simple_cnn(img_batch)  
except Exception as e:
    print(f"\033[91mError during forward pass: {e}\033[0m")

Error during forward pass: mat1 and mat2 shapes cannot be multiplied (28672x14 and 6272x128)


Indeed, the model as provided contains some errors that require debugging.
The message provided by PyTorch when an error occurs can sometimes be cryptic.
It describes that two matrices (`mat1` and `mat2`) cannot be multiplied and provides their shapes.
This indicates that there is a mismatch in the dimensions of the tensors being multiplied, which is a common issue in neural network implementations.

However, the error message does not specify **why and where** in the model the error occurs.
That is when the `forward` method of the model comes into play.
The dynamic graph nature of PyTorch allows you to insert print statements or use debugging tools to inspect the values and shapes of tensors at various points in the `forward` method.

You will define a new class that inherits from the original model and overrides the `forward` method to include print statements that display the shape of the tensor after each layer.
A first try might be to explicitly separate the layers in the `forward` method and, for each layer:
* print the shape of the tensor before the layer (input shape),
* print the shape of some *parameters of the layer* (e.g., weights and biases),
* print the shape of the *activation* tensor after the layer (output shape), which will be the input for the next layer.

You can now run the forward pass again and observe the printed shapes to identify where the mismatch occurs.

In [ ]:
class SimpleCNNDebug(SimpleCNN):
    def __init__(self):
        super().__init__()

    def forward(self,x):
        print(f"Input Shape: {x.shape}")
        print(f"Conv layer parameters shape:",
              self.conv.weight.shape,
              self.conv.bias.shape
                 )
        x = self.relu(self.conv(x))
        print("Shape after conv operation:",
              x.shape)
        
        x = self.pool(x)
        print("Shape after pooling:",
              x.shape)

        print("First FC layer shape:",
              self.fc1.weight.shape,
              self.fc1.bias.shape)
        
        x = self.relu_fc(self.fc1(x))
        print("Shape after first fully conected layer")

        print("2nd FC layer shape:",
              self.fc2.weight.shape,
              self.fc2.bias.shape)
        
        x = self.fc2(x)
        print("Shape after 2nd FC layer:",
              x.shape)

In [16]:
debug_model = SimpleCNNDebug()

try:
    output = debug_model(img_batch)
except Exception as e:
    print(f"Encountered error:\n{e}")

Input Shape: torch.Size([64, 1, 28, 28])
Conv layer parameters shape: torch.Size([32, 1, 3, 3]) torch.Size([32])
Shape after conv operation: torch.Size([64, 32, 28, 28])
Shape after pooling: torch.Size([64, 32, 14, 14])
First FC layer shape: torch.Size([128, 6272]) torch.Size([128])
Encountered error:
mat1 and mat2 shapes cannot be multiplied (28672x14 and 6272x128)


This is already a cleaner output. 
You can already see that all the layers of the convolutional block are working fine, and the shapes are as expected (`batch_size=64` and `out_channels=32`).

**The error occurs in the fully connected block**, specifically at the first linear layer:
`x_pool` has shape `[64, 32, 14, 14]`, but the linear layer expects an input of shape `[64, 2048]` (its weight matrix has shape `[128, 6272]`).

As the linear layer `fc1` expects a 2D input of shape `[batch_size, input_features]`, the `x_pool` is flattened to a 2D tensor with shape `[64*32*14, 14]` before being passed to `fc1`.
This is not the intended shape, and it leads to the dimension mismatch error.

Once you have identified the issue, you can fix it by adding a flattening operation before the first linear layer in the `forward` method.


In [17]:
class SimpleCNNDebug(SimpleCNN):
    def __init__(self):
        super().__init__()

    def forward(self,x):
        print(f"Input Shape: {x.shape}")
        print(f"Conv layer parameters shape:",
              self.conv.weight.shape,
              self.conv.bias.shape
                 )
        x = self.relu(self.conv(x))
        print("Shape after conv operation:",
              x.shape)
        
        x = self.pool(x)
        print("Shape after pooling:",
              x.shape)
        
        ## Flattening before passing to fc layer

        x = torch.flatten(x, start_dim=1)
        print("shape after flattening:",
              x.shape)

        print("First FC layer shape:",
              self.fc1.weight.shape,
              self.fc1.bias.shape)
        
        x = self.relu_fc(self.fc1(x))
        print("Shape after first fully conected layer")

        print("2nd FC layer shape:",
              self.fc2.weight.shape,
              self.fc2.bias.shape)
        
        x = self.fc2(x)
        print("Shape after 2nd FC layer:",
              x.shape)

In [18]:
debug_model = SimpleCNNDebug()

try:
    output = debug_model(img_batch)
except Exception as e:
    print(f"Encountered error:\n{e}")

Input Shape: torch.Size([64, 1, 28, 28])
Conv layer parameters shape: torch.Size([32, 1, 3, 3]) torch.Size([32])
Shape after conv operation: torch.Size([64, 32, 28, 28])
Shape after pooling: torch.Size([64, 32, 14, 14])
shape after flattening: torch.Size([64, 6272])
First FC layer shape: torch.Size([128, 6272]) torch.Size([128])
Shape after first fully conected layer
2nd FC layer shape: torch.Size([10, 128]) torch.Size([10])
Shape after 2nd FC layer: torch.Size([64, 10])


The issue is now fixed, and the model runs without errors! You can see that the shapes of the tensors are as expected after each layer, and the final output has the correct shape of `[64, 10]`, corresponding to the batch size and the number of classes.

Once the model is running without errors, you can jump the next section to refactor the model using `nn.Sequential` for a cleaner and more modular implementation.

## `nn.Sequential` for Modularization

The model is now working correctly, but the `forward` method is quite verbose and repetitive.
To make the code cleaner and more modular, you can use `nn.Sequential` to define the convolutional and fully connected blocks.

In this way you gain several advantages:
* **Modularity**: Each block is defined as a separate module, making it easier to understand and modify.
* **Reusability**: You can easily reuse the blocks in other models or experiments.
* **Cleaner Code**: The `forward` method becomes much simpler, as it only needs to call the blocks sequentially.
* **Less Error-Prone**: By defining the blocks in one place, you reduce the chances of making mistakes when implementing the `forward` method.

In [27]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc_block = nn.Sequential(
            nn.Linear(32*14*14, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):

        x = self.conv_block(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc_block(x)

        return x



In [28]:
cnn_model = CNNModel()
output = cnn_model(img_batch)

print(f"Output shape after passig the input image to model:{output.shape}")

Output shape after passig the input image to model:torch.Size([64, 10])


### Statistical Inspection of the Initialization

A common check when inspecting a model is to look at the statistics of some activations to ensure that they are within a reasonable range.